# Mission 04: Logging Middleware - 실습 노트북

이 노트북은 네 번째 미션을 진행하며 에이전트 실행의 수명 주기(Lifecycle)를 추적하는 LoggingMiddleware를 설계하고 감사 로그를 파일로 적재하는 법을 실습합니다.

In [ ]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from app.tools import web_search

### [미션 1] LoggingMiddleware 스켈레톤 구현하기

아래 빈 칸에 `before_agent`, `after_agent`, `wrap_tool_call`을 알맞게 구현하세요.

In [ ]:
import time
import json
from typing import Any, Dict
from langchain.agents.middleware import AgentMiddleware

class StudentLoggingMiddleware(AgentMiddleware):
    def __init__(self, log_dir="./artifacts/logs"):
        self.log_dir = log_dir
        os.makedirs(self.log_dir, exist_ok=True)
        # TODO: 런타임별(요청별) 고유 정보를 격리 저장할 스레드/비동기 세이프한 저장소 생성
        # self._active_runs = {}
        
    def before_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        # TODO: 시작 시간(time.time())과 사용자 질문 쿼리를 기록(active_runs에 저장)하고,
        # 콘솔에 시작 로그(예: 🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===)를 이쁘게 출력하세요.
        return None
        
    def after_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        # TODO: active_runs에서 해당 runtime 객체의 기록을 pop하고,
        # 소요 시간(ms)을 계산한 뒤, 대화 최종 답변 및 대화 궤적(dialogue_history)을 추출하여
        # session_id를 취득하고 _append_log를 호출해 세션별 JSON라인 파일에 감사 로그를 적재하세요.
        return None
        
    def wrap_tool_call(self, request, handler):
        logging_enabled = getattr(request.runtime.context, "logging_enabled", False) if request.runtime and request.runtime.context else False
        if not logging_enabled:
            return handler(request)
            
        # TODO: 도구 실행(handler(request)) 전후의 수행 시간을 재어 각 도구별 수행 시간을 계산하고,
        # session_id에 상응하는 감사 로그 파일에 도구 실행 정보를 기록하고 실행 결과(response)를 반환하세요.
        return handler(request)
        
    def _append_log(self, session_id, log_data):
        # TODO: 감사 로그 지정 디렉토리 내에 세션별 JSON라인 파일로 적재하는 코드를 작성하세요.
        # log_file = os.path.join(self.log_dir, f"{session_id}.jsonl")
        pass

### [미션 2] 미들웨어가 연동된 에이전트 생성 및 검증

작성한 미들웨어를 create_agent에 등록하고 대화를 요청하여 로그가 정확히 누적되는지 확인하세요.

In [ ]:
log_directory = "./artifacts/logs"
thread_id = "session_logging_test_04"
log_filepath = os.path.join(log_directory, f"{thread_id}.jsonl")

if os.path.exists(log_filepath):
    os.remove(log_filepath)

logging_middleware = StudentLoggingMiddleware(log_dir=log_directory)
llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)

from app.prompts import CHATBOT_SYSTEM_PROMPT

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    middleware=[logging_middleware],
    context_schema=AgentContext
)

context_obj = AgentContext(logging_enabled=True)

res = agent.invoke(
    {"messages": [HumanMessage(content="마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)

### [미션 3] 생성된 감사 로그 검증

실제로 지정된 경로에 JSON 라인으로 로그들이 정상 저장되었는지 프린트해봅니다.

In [ ]:
if os.path.exists(log_filepath):
    print(f"📝 적재된 로그 내용 ({log_filepath}):")
    print("-" * 80)
    with open(log_filepath, "r", encoding="utf-8") as f:
        for line in f:
            print(line.strip())
    print("-" * 80)
else:
    print("❌ 로그 파일이 생성되지 않았습니다.")